<a href="https://colab.research.google.com/github/LunaTic-Neon/2026-1-NLP/blob/main/26_1_0320_NLP_w3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install datasets konlpy
from datasets import load_dataset

# KLUE 데이터셋의 ynat 로드
dataset = load_dataset("klue", "ynat")

# train 데이터에서 title의 value 1000개 추출
documents = dataset['train']['title'][:1000]

# 데이터 확인
print(f"문서 개수: {len(documents)}")
print(f"첫 번째 문서: {documents[0]}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 29.1 MB/s eta 0:00:00
문서 개수: 1000
첫 번째 문서: 유튜브 내달 2일까지 크리에이터 지원 공간 운영


In [4]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import warnings
warnings.filterwarnings('ignore')

okt = Okt()

def my_tokenizer(doc):
    return [token for token, pos in okt.pos(doc) if pos in ['Noun', 'Verb', 'Adjective']]

# 1. BOW 기반의 Count DTM 생성
cv = CountVectorizer(max_features=1000, tokenizer=my_tokenizer)
dtm_cv = cv.fit_transform(documents)

# 2. TF-IDF 기반의 DTM 생성
transformer = TfidfTransformer()
dtm_tfidf = transformer.fit_transform(dtm_cv)

print(f"Count DTM 형태: {dtm_cv.shape}")
print(f"TF-IDF DTM 형태: {dtm_tfidf.shape}")

Count DTM 형태: (1000, 1000)
TF-IDF DTM 형태: (1000, 1000)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 기준 문서
source_cv = dtm_cv[0]
source_tfidf = dtm_tfidf[0]

# 코사인 유사도 계산
sim_cv = cosine_similarity(source_cv, dtm_cv)
sim_tfidf = cosine_similarity(source_tfidf, dtm_tfidf)

# 자신 포함 가장 유사도가 높은 상위 10개의 인덱스 추출
top_10_cv_indices = np.argsort(sim_cv[0])[::-1][:10]
top_10_tfidf_indices = np.argsort(sim_tfidf[0])[::-1][:10]

print("-" * 70 + "\n")
print(f"[기준 문서]: {documents[0]}\n")

print("[BOW (CountVectorizer) 기반 가장 유사한 문서 Top 10]")
for rank, idx in enumerate(top_10_cv_indices, 1):
    print(f"{rank}위 (인덱스 {idx}, 유사도: {sim_cv[0][idx]:.4f}): {documents[idx]}")

print("\n" + "-" * 70 + "\n")

print("[TF-IDF (TfidfTransformer) 기반 가장 유사한 문서 Top 10]")
for rank, idx in enumerate(top_10_tfidf_indices, 1):
    print(f"{rank}위 (인덱스 {idx}, 유사도: {sim_tfidf[0][idx]:.4f}): {documents[idx]}")

----------------------------------------------------------------------

[기준 문서]: 유튜브 내달 2일까지 크리에이터 지원 공간 운영

[BOW (CountVectorizer) 기반 가장 유사한 문서 Top 10]
1위 (인덱스 0, 유사도: 1.0000): 유튜브 내달 2일까지 크리에이터 지원 공간 운영
2위 (인덱스 390, 유사도: 0.3162): 공간에 겹쳐 새긴 역사와 기억…서효인 시집 여수
3위 (인덱스 982, 유사도: 0.3162): 장마 일주일 지각 내달 2일부터 중부 본격적인 장맛비
4위 (인덱스 271, 유사도: 0.2582): 성신양회·GS글로벌 미얀마 시멘트부두 추진…해수부 지원
5위 (인덱스 244, 유사도: 0.2582): 넷마블 신작 킹 오브 파이터 올스타 내달 9일 출시
6위 (인덱스 988, 유사도: 0.2582): 금융소비자원 DLS·DLF 피해자 상담센터 운영
7위 (인덱스 151, 유사도: 0.2236): 유튜브 1분 넘는 광고도 소비자가 주목한다
8위 (인덱스 150, 유사도: 0.2236): 野 전경련 어버이연합 뒷돈 지원 의혹 국회 진상조사
9위 (인덱스 688, 유사도: 0.2236): 유튜브 동영상 외에 음악감상용으로도 가장 많이 쓴다
10위 (인덱스 105, 유사도: 0.2236): 사우디 이라크 스포츠도시 건설에 1조원 지원

----------------------------------------------------------------------

[TF-IDF (TfidfTransformer) 기반 가장 유사한 문서 Top 10]
1위 (인덱스 0, 유사도: 1.0000): 유튜브 내달 2일까지 크리에이터 지원 공간 운영
2위 (인덱스 390, 유사도: 0.3791): 공간에 겹쳐 새긴 역사와 기억…서효인 시집 여수
3위 (인덱스 988, 유사도: 0.2878): 금융소비자원 DLS·DLF 피해자 상담센터 운영
4위 (인덱스 982, 유사도: 0